# Protocol - A: Driver Behaviour Analysis

In [1]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LSTM,
    TimeDistributed,
    GlobalAveragePooling2D,
    GlobalAveragePooling1D,
    MultiHeadAttention,
    LayerNormalization,
    Add,
    Embedding,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

try:
    from tensorflow.keras.optimizers.legacy import Adam
except:
    from tensorflow.keras.optimizers import Adam

# =====================================================
# CONFIG
# =====================================================

TRAIN_PATH = '/Users/ishashetye/Desktop/Be_Safe_On_the_Road/state-farm-distracted-driver-detection/imgs/train'
CSV_PATH   = '/Users/ishashetye/Desktop/Be_Safe_On_the_Road/state-farm-distracted-driver-detection/driver_imgs_list.csv'

IMAGE_SIZE = (160, 160)
FRAME_SEQ  = 5
NUM_CLASSES = 10
EPOCHS = 30
BATCH_SIZE = 16
SEED = 42

MAX_SEQS_PER_CLASS = 200

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

LABEL_MAP = {
    'c0': 'safe driving',
    'c1': 'texting - right',
    'c2': 'talking on the phone - right',
    'c3': 'texting - left',
    'c4': 'talking on the phone - left',
    'c5': 'operating the radio',
    'c6': 'drinking',
    'c7': 'reaching behind',
    'c8': 'hair and makeup',
    'c9': 'talking to passenger',
}

CLASS_NAMES = list(LABEL_MAP.values())

# =====================================================
# DRIVER-WISE SPLIT
# =====================================================

print('\nLoading driver metadata...')

df = pd.read_csv(CSV_PATH)

all_drivers = sorted(df['subject'].unique())

train_drivers, temp_drivers = train_test_split(
    all_drivers,
    test_size=0.30,
    random_state=SEED,
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.50,
    random_state=SEED,
)

print(f'Train drivers: {len(train_drivers)}')
print(f'Val drivers:   {len(val_drivers)}')
print(f'Test drivers:  {len(test_drivers)}')

# =====================================================
# SEQUENCE CREATION
# =====================================================

def build_sequences(split_drivers):
    X = []
    y = []

    for label in LABEL_MAP:

        class_df = df[
            (df['classname'] == label) &
            (df['subject'].isin(split_drivers))
        ]

        files = class_df['img'].tolist()

        random.shuffle(files)

        files = files[:FRAME_SEQ * MAX_SEQS_PER_CLASS]

        for i in range(0, len(files) - FRAME_SEQ + 1, FRAME_SEQ):

            seq = []
            valid = True

            for j in range(FRAME_SEQ):

                img_name = files[i + j]
                img_path = os.path.join(TRAIN_PATH, label, img_name)

                img = cv2.imread(img_path)

                if img is None:
                    valid = False
                    break

                img = cv2.resize(img, IMAGE_SIZE)
                img = img.astype(np.float32) / 255.0

                seq.append(img)

            if valid:
                X.append(seq)
                y.append(int(label[-1]))

    X = np.asarray(X, dtype=np.float32)
    y = to_categorical(np.asarray(y), NUM_CLASSES)

    return X, y

print('\nBuilding train sequences...')
X_train, y_train = build_sequences(train_drivers)

print('\nBuilding validation sequences...')
X_val, y_val = build_sequences(val_drivers)

print('\nBuilding test sequences...')
X_test, y_test = build_sequences(test_drivers)

print('\nDataset Shapes')
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

# =====================================================
# COMMON FEATURE EXTRACTOR
# =====================================================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
)

base_model.trainable = False

# =====================================================
# TRANSFORMER BLOCK
# =====================================================

def transformer_block(x, num_heads=2, key_dim=32, ff_dim=128, dropout=0.2):

    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout,
    )(x, x)

    x1 = LayerNormalization(epsilon=1e-6)(Add()([x, attn]))

    ff = Dense(ff_dim, activation='relu')(x1)
    ff = Dense(x.shape[-1])(ff)

    x2 = LayerNormalization(epsilon=1e-6)(Add()([x1, ff]))

    return x2

# =====================================================
# MODEL 1 - FAIR CNN BASELINE
# =====================================================

def build_cnn_baseline():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = tf.reduce_mean(x, axis=1)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='CNN_Baseline')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# MODEL 2 - LSTM
# =====================================================

def build_lstm():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = LSTM(128)(x)

    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='LSTM_Model')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# MODEL 3 - TEMPORAL TRANSFORMER
# =====================================================

def build_transformer():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = TimeDistributed(Dense(128, activation='relu'))(x)

    positions = tf.range(start=0, limit=FRAME_SEQ, delta=1)

    x = x + Embedding(
        input_dim=FRAME_SEQ,
        output_dim=128,
    )(positions)

    x = transformer_block(x)

    x = GlobalAveragePooling1D()(x)

    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='Transformer_Model')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# TRAINING HELPER
# =====================================================

def train_model(model, name):

    print('\n' + '=' * 60)
    print(name)
    print('=' * 60)

    es = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=1,
    )

    return history

# =====================================================
# EVALUATION
# =====================================================

def evaluate(model, name):

    pred = np.argmax(model.predict(X_test), axis=1)
    true = np.argmax(y_test, axis=1)

    acc = accuracy_score(true, pred)
    prec = precision_score(true, pred, average='macro')
    rec = recall_score(true, pred, average='macro')
    f1 = f1_score(true, pred, average='macro')

    print('\n' + '=' * 60)
    print(name)
    print('=' * 60)

    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1 Score : {f1:.4f}')

    print('\nClassification Report\n')

    print(classification_report(
        true,
        pred,
        target_names=CLASS_NAMES,
    ))

    return {
        'Method': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
    }

# =====================================================
# MAIN EXPERIMENTS
# =====================================================

results = []

# -----------------------------------------------------
# CNN BASELINE
# -----------------------------------------------------

cnn_model = build_cnn_baseline()
cnn_model.summary()

train_model(cnn_model, 'CNN Baseline')

results.append(
    evaluate(cnn_model, 'CNN Baseline')
)

# -----------------------------------------------------
# LSTM
# -----------------------------------------------------

lstm_model = build_lstm()
lstm_model.summary()

train_model(lstm_model, 'LSTM Model')

results.append(
    evaluate(lstm_model, 'LSTM Model')
)

# -----------------------------------------------------
# TRANSFORMER
# -----------------------------------------------------

transformer_model = build_transformer()
transformer_model.summary()

train_model(transformer_model, 'Transformer Model')

results.append(
    evaluate(transformer_model, 'Transformer Model')
)

# =====================================================
# FINAL COMPARISON TABLE
# =====================================================

results_df = pd.DataFrame(results)

print('\nFINAL RESULTS\n')
print(results_df)

results_df.to_csv('fair_comparison_results.csv', index=False)

print('\nSaved: fair_comparison_results.csv')



Loading driver metadata...
Train drivers: 18
Val drivers:   4
Test drivers:  4

Building train sequences...

Building validation sequences...

Building test sequences...

Dataset Shapes
(2000, 5, 160, 160, 3) (2000, 10)
(569, 5, 160, 160, 3) (569, 10)
(729, 5, 160, 160, 3) (729, 10)
Model: "CNN_Baseline"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 5, 160, 160, 3)   0         
                             ]                                   
                                                                 
 time_distributed (TimeDist  (None, 5, 5, 5, 1280)     2257984   
 ributed)                                                        
                                                                 
 time_distributed_1 (TimeDi  (None, 5, 1280)           0         
 stributed)                                                      
                                 

In [2]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LSTM,
    TimeDistributed,
    GlobalAveragePooling2D,
    GlobalAveragePooling1D,
    MultiHeadAttention,
    LayerNormalization,
    Add,
    Embedding,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

try:
    from tensorflow.keras.optimizers.legacy import Adam
except:
    from tensorflow.keras.optimizers import Adam

# =====================================================
# CONFIG
# =====================================================

TRAIN_PATH = '/Users/ishashetye/Desktop/Be_Safe_On_the_Road/state-farm-distracted-driver-detection/imgs/train'
CSV_PATH   = '/Users/ishashetye/Desktop/Be_Safe_On_the_Road/state-farm-distracted-driver-detection/driver_imgs_list.csv'

IMAGE_SIZE = (160, 160)
FRAME_SEQ  = 5
NUM_CLASSES = 10
EPOCHS = 30
BATCH_SIZE = 16
SEED = 42

MAX_SEQS_PER_CLASS = 200

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

LABEL_MAP = {
    'c0': 'safe driving',
    'c1': 'texting - right',
    'c2': 'talking on the phone - right',
    'c3': 'texting - left',
    'c4': 'talking on the phone - left',
    'c5': 'operating the radio',
    'c6': 'drinking',
    'c7': 'reaching behind',
    'c8': 'hair and makeup',
    'c9': 'talking to passenger',
}

CLASS_NAMES = list(LABEL_MAP.values())

# =====================================================
# DRIVER-WISE SPLIT
# =====================================================

print('\nLoading driver metadata...')

df = pd.read_csv(CSV_PATH)

all_drivers = sorted(df['subject'].unique())

train_drivers, temp_drivers = train_test_split(
    all_drivers,
    test_size=0.30,
    random_state=SEED,
)

val_drivers, test_drivers = train_test_split(
    temp_drivers,
    test_size=0.50,
    random_state=SEED,
)

print(f'Train drivers: {len(train_drivers)}')
print(f'Val drivers:   {len(val_drivers)}')
print(f'Test drivers:  {len(test_drivers)}')

# =====================================================
# SEQUENCE CREATION
# =====================================================

# =====================================================
# SEQUENCE CREATION (driver-grouped, temporally ordered)
# =====================================================

def build_sequences(split_drivers):
    X = []
    y = []

    for label in LABEL_MAP:

        class_df = df[
            (df['classname'] == label) &
            (df['subject'].isin(split_drivers))
        ]

        # Collect sequences per driver so frames in one sequence
        # come from the same person doing the same activity.
        # Sort by filename so consecutive frames stay adjacent.
        driver_sequences = []

        for driver in class_df['subject'].unique():

            driver_files = sorted(
                class_df[class_df['subject'] == driver]['img'].tolist()
            )

            for i in range(0, len(driver_files) - FRAME_SEQ + 1, FRAME_SEQ):
                driver_sequences.append(
                    (driver, driver_files[i:i + FRAME_SEQ])
                )

        # Shuffle the *sequences* (not the frames inside them)
        # so the cap across drivers is fair, then apply the cap.
        random.shuffle(driver_sequences)
        driver_sequences = driver_sequences[:MAX_SEQS_PER_CLASS]

        for _, seq_files in driver_sequences:

            seq = []
            valid = True

            for img_name in seq_files:

                img_path = os.path.join(TRAIN_PATH, label, img_name)

                img = cv2.imread(img_path)

                if img is None:
                    valid = False
                    break

                img = cv2.resize(img, IMAGE_SIZE)
                img = img.astype(np.float32) / 255.0

                seq.append(img)

            if valid:
                X.append(seq)
                y.append(int(label[-1]))

    X = np.asarray(X, dtype=np.float32)
    y = to_categorical(np.asarray(y), NUM_CLASSES)

    return X, y

print('\nBuilding train sequences...')
X_train, y_train = build_sequences(train_drivers)

print('\nBuilding validation sequences...')
X_val, y_val = build_sequences(val_drivers)

print('\nBuilding test sequences...')
X_test, y_test = build_sequences(test_drivers)

print('\nDataset Shapes')
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

# =====================================================
# COMMON FEATURE EXTRACTOR
# =====================================================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
)

base_model.trainable = False

# =====================================================
# TRANSFORMER BLOCK
# =====================================================

def transformer_block(x, num_heads=2, key_dim=32, ff_dim=128, dropout=0.2):

    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout,
    )(x, x)

    x1 = LayerNormalization(epsilon=1e-6)(Add()([x, attn]))

    ff = Dense(ff_dim, activation='relu')(x1)
    ff = Dense(x.shape[-1])(ff)

    x2 = LayerNormalization(epsilon=1e-6)(Add()([x1, ff]))

    return x2

# =====================================================
# MODEL 1 - FAIR CNN BASELINE
# =====================================================

def build_cnn_baseline():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = tf.reduce_mean(x, axis=1)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='CNN_Baseline')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# MODEL 2 - LSTM
# =====================================================

def build_lstm():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = LSTM(128)(x)

    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='LSTM_Model')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# MODEL 3 - TEMPORAL TRANSFORMER
# =====================================================

def build_transformer():

    inp = Input(shape=(FRAME_SEQ, IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = TimeDistributed(base_model)(inp)
    x = TimeDistributed(GlobalAveragePooling2D())(x)

    x = TimeDistributed(Dense(128, activation='relu'))(x)

    positions = tf.range(start=0, limit=FRAME_SEQ, delta=1)

    x = x + Embedding(
        input_dim=FRAME_SEQ,
        output_dim=128,
    )(positions)

    x = transformer_block(x)

    x = GlobalAveragePooling1D()(x)

    x = Dropout(0.5)(x)

    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out, name='Transformer_Model')

    model.compile(
        optimizer=Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return model

# =====================================================
# TRAINING HELPER
# =====================================================

def train_model(model, name):

    print('\n' + '=' * 60)
    print(name)
    print('=' * 60)

    es = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=1,
    )

    return history

# =====================================================
# EVALUATION
# =====================================================

def evaluate(model, name):

    pred = np.argmax(model.predict(X_test), axis=1)
    true = np.argmax(y_test, axis=1)

    acc = accuracy_score(true, pred)
    prec = precision_score(true, pred, average='macro')
    rec = recall_score(true, pred, average='macro')
    f1 = f1_score(true, pred, average='macro')

    print('\n' + '=' * 60)
    print(name)
    print('=' * 60)

    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1 Score : {f1:.4f}')

    print('\nClassification Report\n')

    print(classification_report(
        true,
        pred,
        target_names=CLASS_NAMES,
    ))

    return {
        'Method': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
    }

# =====================================================
# MAIN EXPERIMENTS
# =====================================================

results = []

# -----------------------------------------------------
# CNN BASELINE
# -----------------------------------------------------

cnn_model = build_cnn_baseline()
cnn_model.summary()

train_model(cnn_model, 'CNN Baseline')

results.append(
    evaluate(cnn_model, 'CNN Baseline')
)

# -----------------------------------------------------
# LSTM
# -----------------------------------------------------

lstm_model = build_lstm()
lstm_model.summary()

train_model(lstm_model, 'LSTM Model')

results.append(
    evaluate(lstm_model, 'LSTM Model')
)

# -----------------------------------------------------
# TRANSFORMER
# -----------------------------------------------------

transformer_model = build_transformer()
transformer_model.summary()

train_model(transformer_model, 'Transformer Model')

results.append(
    evaluate(transformer_model, 'Transformer Model')
)

# =====================================================
# FINAL COMPARISON TABLE
# =====================================================

results_df = pd.DataFrame(results)

print('\nFINAL RESULTS\n')
print(results_df)

results_df.to_csv('fair_comparison_results.csv', index=False)

print('\nSaved: fair_comparison_results.csv')



Loading driver metadata...
Train drivers: 18
Val drivers:   4
Test drivers:  4

Building train sequences...

Building validation sequences...

Building test sequences...

Dataset Shapes
(2000, 5, 160, 160, 3) (2000, 10)
(559, 5, 160, 160, 3) (559, 10)
(716, 5, 160, 160, 3) (716, 10)
Model: "CNN_Baseline"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 5, 160, 160, 3)   0         
                             ]                                   
                                                                 
 time_distributed (TimeDist  (None, 5, 5, 5, 1280)     2257984   
 ributed)                                                        
                                                                 
 time_distributed_1 (TimeDi  (None, 5, 1280)           0         
 stributed)                                                      
                                 